# Imports and Setup


In [ ]:

import pandas as pd
import numpy as np
from sodapy import Socrata

APP_KEY = "t2GPRE1GXlktxvqOIFMelPIvT"

version = "4"

client = Socrata(
    "data.cityofnewyork.us",
    APP_KEY,
    timeout=90
)

RELEASE_NAMES = ["prelim", "exec", "adopt"]

# Expense Budget


https://data.cityofnewyork.us/api/v3/views/mwzb-yiwb/query.json


In [ ]:
EXPENSE_DATASET_ID = "mwzb-yiwb"

### Check Fiscal Year


In [ ]:
# 1. Fetch the 2 most recent unique fiscal years present in the dataset
year_query = "SELECT fiscal_year GROUP BY fiscal_year ORDER BY fiscal_year DESC LIMIT 2"
current_fy_json = client.get(EXPENSE_DATASET_ID, query=year_query)

In [ ]:
# Extract the years into a list, e.g., ['2026', '2025']
planned_fy = [row['fiscal_year'] for row in current_fy_json][0]

planned_fy_name = f"FY{int(planned_fy) % 100}"

print(planned_fy)
print(planned_fy_name)

print()

current_fy = int(planned_fy) - 1
current_fy_name = f"FY{int(current_fy) % 100}"

print(current_fy)
print(current_fy_name)

In [ ]:
where_string = f"fiscal_year IN ({planned_fy})"

print(f"where_string:\n{where_string}\n")

orderby_arr = [
    "agency_number", # 2: 
    "unit_appropriation_number", # 4: 
    "responsibility_center_code", # 23: 
    "budget_code_number", # 6: 
    "object_class_number", # 8: 
    "object_code", # 10: 
    "publication_date DESC"
]

orderby_string = ", ".join(orderby_arr)

print(f"orderby_string:\n{orderby_string}\n")

### Query Data if necessary


In [ ]:
data = []
offset = 0

limit = 1000

try:
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")
except:
    print(f"Querying for fiscal year {planned_fy}_v{version}:\n")
    while True:
        print(f"Fetching rows starting at offset: {offset}")

        temp = client.get(
            dataset_identifier=EXPENSE_DATASET_ID,
            # select=select_string,
            where=where_string,
            # group=groupby_string,
            order=orderby_string,
            # order=":id",
            limit=limit,
            offset=offset
        )
        
        if not temp:
            break
        
        # print(temp[0])
        
        data.extend(temp)
        
        offset += limit
        
        # break
            
    print(f"End of query. Successfully fetched {len(data)} total rows.")
    
    len(data)
    
    # Convert to pandas DataFrame
    df = pd.DataFrame.from_records(data)
    # df.drop(columns=["financial_plan_savings_flag"], inplace=True)
    
    df.to_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv", index=False)
    df = pd.read_csv(f"./.data_cache/backup_{planned_fy}_v{version}.csv")

In [ ]:
len(df)

## Sort and reorder


In [ ]:
for i, col in enumerate(df.columns):
    print(f"{i}: '{col}',")

In [ ]:
numeric_cols = [
    "adopted_budget_amount",
    "current_modified_budget_amount",
    "financial_plan_amount",
    
    "adopted_budget_position",
    "current_modified_budget_position",
    "financial_plan_position",
    
    "adopted_budget_number_of_contracts",
    "current_modified_budget_number_of_contracts",
    "financial_plan_number_of_contracts"
]

categorical_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    # "financial_plan_savings_flag",
    # "intra_city_purchase_code",
    # "personal_service_other_than_personal_service_indicator",
]

In [ ]:
arr = list(df.columns)
for col in numeric_cols:
    arr.remove(col)
    
arr.remove("fiscal_year")
print(arr)

In [ ]:
df[arr] = df[arr].fillna("(blank)")

In [ ]:
for cat_col in categorical_cols:
    if cat_col in df.columns:
        df[cat_col] = df[cat_col].apply(lambda x: str.upper(x) if type(x) == str else x)

In [ ]:
df = df.sort_values(
    [
        "agency_number",
        "unit_appropriation_number",
        "responsibility_center_code",
        "budget_code_number",
        "object_class_number",
        "object_code",
        "publication_date"
        ],
    ascending=[True, True, True, True, True, True, True]
    ).reset_index(drop=True)

df

In [ ]:
# Reorder columns
df = df[
    [
    "publication_date", # 0: 
    "fiscal_year", # 1: 
    "agency_number", # 2: 
    "agency_name", # 3: 
    "unit_appropriation_number", # 4: 
    "unit_appropriation_name", # 5: 
    "responsibility_center_code", # 23: 
    "responsibility_center_name", # 24: 
    "budget_code_number", # 6: 
    "budget_code_name", # 7: 
    "object_class_number", # 8: 
    "object_class_name", # 9: 
    "object_code", # 10: 
    "object_code_name", # 11: 
    # "intra_city_purchase_code", # 25: 
    # "personal_service_other_than_personal_service_indicator", # 12: 
    # "financial_plan_savings_flag", # 13: 
    "adopted_budget_amount", # 14: 
    "current_modified_budget_amount", # 15: 
    "financial_plan_amount", # 16: 
    "adopted_budget_position", # 17: 
    "current_modified_budget_position", # 18: 
    "financial_plan_position", # 19: 
    "adopted_budget_number_of_contracts", # 20: 
    "current_modified_budget_number_of_contracts", # 21: 
    "financial_plan_number_of_contracts", # 22: 
    ]
]
df

## Compare against 2027 master


In [ ]:
master_2027_df = pd.read_excel("./2027/master_2027.xlsx", header=3)
master_2027_df_exec = master_2027_df[master_2027_df['publication_date'] == 20260512]

In [ ]:
master_2027_df_exec

In [ ]:
master_2027_df_exec.columns

In [ ]:
for cat_col in categorical_cols:
    if cat_col in master_2027_df_exec.columns:
        master_2027_df_exec[cat_col] = master_2027_df_exec[cat_col].apply(lambda x: str.upper(x) if type(x) == str else x)

In [ ]:
df_pivot = df.groupby([
    'publication_date', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'responsibility_center_code', 'responsibility_center_name',
       'budget_code_number', 'budget_code_name', 'object_class_number',
       'object_class_name', 'object_code', 'object_code_name',
       ], dropna=False).sum().reset_index()

df_pivot = df_pivot[[
                     'publication_date', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'responsibility_center_code', 'responsibility_center_name',
       'budget_code_number', 'budget_code_name', 'object_class_number',
       'object_class_name', 'object_code', 'object_code_name',
       'adopted_budget_amount',
       'current_modified_budget_amount',
       'financial_plan_amount'
       ]]

df_pivot = df_pivot[df_pivot['publication_date'] == 20260512]

df_pivot

In [ ]:
# 1. Left join with indicator=True
merged = df_pivot.merge(master_2027_df_exec, on=[
    'publication_date', 'agency_number', 'agency_name',
       'unit_appropriation_number', 'unit_appropriation_name',
       'responsibility_center_code', 'responsibility_center_name',
       'budget_code_number', 'budget_code_name', 'object_class_number',
       'object_class_name', 'object_code', 'object_code_name',
    ], how='outer', indicator=True)


merged

In [ ]:
merged[(merged["adopted_budget_amount"] != merged["sum_of_adopted_budget_amount"]) | (merged["current_modified_budget_amount"] != merged["sum_of_current_modified_budget_amount"]) | (merged["financial_plan_amount"] != merged["sum_of_financial_plan_amount"])]

In [ ]:
rows_not_in_both = merged[merged['_merge'] != 'both']
print(rows_not_in_both['_merge'].value_counts())

rows_not_in_both

## Track releases


In [ ]:
pub_dates = df["publication_date"].sort_values(ascending=True).unique().tolist()[:]

num_pubs_this_year = len(pub_dates)

print(f"{num_pubs_this_year} pub_dates in FY {planned_fy}: {pub_dates}")

In [ ]:
df.to_excel(f"current_api_{planned_fy}.xlsx",sheet_name="Raw",index=False)

## Set up base DataFrame and map function


In [ ]:
base_df = df[categorical_cols].drop_duplicates().reset_index(drop=True)
len(base_df)

## Loop through each release


In [ ]:
print(pub_dates)

print(planned_fy)

for i, pub_date in enumerate(pub_dates):
    release_name = RELEASE_NAMES[i]
    
    ith_release_df = df[df['publication_date'] == pub_date]
    
    print(i, release_name)

## Collapse Releases


In [ ]:
collapsed_df = base_df.copy()

new_numeric_cols = []


start_of_numerical_cols = len(categorical_cols)

# collapsed_df.head()

for i, pub_date in enumerate(pub_dates):
    print(f"[{i}/{num_pubs_this_year}] pub_date -- {pub_date} ({RELEASE_NAMES[i]}):")
    
    ith_release_df = df[df["publication_date"] == pub_date][categorical_cols + numeric_cols].copy()
    
    print(len(ith_release_df))
    
    # print(i)
    
    # print(ith_release_df.columns[len(categorical_cols):])
    
    if i <= 2:
        for col in ["financial_plan_amount", "financial_plan_position", "financial_plan_number_of_contracts"]:
            # print(col)
            ith_release_df = ith_release_df.rename(
                columns={
                    f"{col}":f"{col}_{RELEASE_NAMES[i]}",
                    }
            )
            # print()
            # print(ith_release_df.columns)
    
        if i < num_pubs_this_year - 1:
            ith_release_df = ith_release_df.drop(columns=[
                f"adopted_budget_amount",
                f"current_modified_budget_amount",
                
                f"adopted_budget_position",
                f"current_modified_budget_position",
                
                f"adopted_budget_number_of_contracts",
                f"current_modified_budget_number_of_contracts",
                ])
            
        print(f"Adding new numeric columns:{ith_release_df.columns[start_of_numerical_cols:]}\n")
        new_numeric_cols.extend(ith_release_df.columns[start_of_numerical_cols:])
    else:
        raise Exception(f"Bad indexing, i:{i} >= num_pubs_this_year or i:{i} < 0")
    
    # print(ith_release_df.columns)
    print()
    
    collapsed_df = collapsed_df.merge(right=ith_release_df,on=categorical_cols, how='left')
    
    # break

collapsed_df = collapsed_df.reset_index(drop=True)


print(len(base_df))
print(len(collapsed_df))


collapsed_df.columns

In [ ]:
new_numeric_cols

In [ ]:
collapsed_df

In [ ]:
collapsed_df[collapsed_df.duplicated(keep=False)]

## Object Code Level


In [ ]:
Object_Code_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    "object_class_number",
    "object_class_name",
    "object_code",
    "object_code_name",
    # "financial_plan_savings_flag",
]

main_cols = [
    'adopted_budget_amount',
    'current_modified_budget_amount',
    'financial_plan_amount_exec'
]

object_code_collapsed_df = collapsed_df.groupby(Object_Code_cols,dropna=False).sum().reset_index()

object_code_collapsed_df = object_code_collapsed_df[Object_Code_cols + new_numeric_cols]

object_code_collapsed_df[Object_Code_cols + main_cols]


In [ ]:
# object_code_collapsed_df[object_code_collapsed_df.duplicated(keep=False)]

## Budget Code Level


In [ ]:
Budget_Code_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    "budget_code_number",
    "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

main_cols = [
    'adopted_budget_amount',
    'current_modified_budget_amount',
    'financial_plan_amount_exec'
]

budget_code_collapsed_df = collapsed_df[Budget_Code_cols + new_numeric_cols].groupby(Budget_Code_cols,dropna=False).sum().reset_index()

# budget_code_collapsed_df = budget_code_collapsed_df[Budget_Code_cols + new_numeric_cols]

# budget_code_collapsed_df[Budget_Code_cols + main_cols]

budget_code_collapsed_df

In [ ]:
# budget_code_collapsed_df[budget_code_collapsed_df.duplicated(keep=False)]

## Responsibility Center Level


In [ ]:
RC_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    "responsibility_center_code",
    "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

RC_collapsed_df = collapsed_df[RC_cols + new_numeric_cols].groupby(RC_cols,dropna=False).sum().reset_index()

# RC_collapsed_df = RC_collapsed_df[RC_cols + new_numeric_cols]

RC_collapsed_df
# RC_collapsed_df[RC_cols + main_cols]
# RC_collapsed_df[RC_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][RC_cols + main_cols]

In [ ]:
RC_collapsed_df[RC_collapsed_df.duplicated(keep=False)]

## Unit of Appropriation Level


In [ ]:
UoA_cols = [
    "agency_number",
    "agency_name",
    "unit_appropriation_number",
    "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

UoA_collapsed_df = collapsed_df[UoA_cols + new_numeric_cols].groupby(UoA_cols,dropna=False).sum().reset_index()

# UoA_collapsed_df = UoA_collapsed_df[UoA_cols + new_numeric_cols]

UoA_collapsed_df
# UoA_collapsed_df[UoA_cols + main_cols]
# UoA_collapsed_df[UoA_collapsed_df['agency_name'] == "DEPARTMENT OF EDUCATION"][UoA_cols + main_cols]

In [ ]:
# UoA_collapsed_df[UoA_collapsed_df.duplicated(keep=False)]

## Agency Level


In [ ]:
Agency_cols = [
    "agency_number",
    "agency_name",
    # "unit_appropriation_number",
    # "unit_appropriation_name",
    # "responsibility_center_code",
    # "responsibility_center_name",
    # "budget_code_number",
    # "budget_code_name",
    # "object_class_number",
    # "object_class_name",
    # "object_code",
    # "object_code_name",
    # "financial_plan_savings_flag",
]

Agency_collapsed_df = collapsed_df[Agency_cols + new_numeric_cols].groupby(Agency_cols,dropna=False).sum().reset_index()

# Agency_collapsed_df = Agency_collapsed_df[Agency_cols + new_numeric_cols]

# Agency_collapsed_df[Agency_cols + main_cols]
Agency_collapsed_df